In [1]:
"""Plots the BWB VSPAERO model of the median-HV run's non-dominated front.

Picks three representative designs off the true robust ND front of the
median-hypervolume run (Results/BWB_VSP_AERO/Hypervolume/hypervolume_results.json
identifies which run that is - currently run 17, the only valid run):
the min-CD extreme, the max-CL extreme, and the knee point (minimum
normalized distance to the ideal point). Each is re-run through the real
VSPAERO VLM solver (not the surrogate) at the design cruise condition, with
CpSlicer swept spanwise to recover a genuine surface pressure-coefficient
field. Local surface velocity ratio V/Vinf is then derived from that Cp via
the standard compressible (isentropic) Bernoulli relation - VSPAERO's VLM
solver has no off-body flow field of its own, only circulation/Cp on the
thin lifting surface, so this is the honest way to get a "velocity contour"
out of it.

Produces, under optimization/bwb_cfd_median_hv/:
  bwb_<label>.npz                       - cached per-design Cp slice data
  BWB_run17_front_highlighted.png       - ND front with the 3 points marked
                                           (CL axis kept as the raw -CL
                                           objective, matching F[:,1])
  BWB_run17_velocity_ratio.png          - 3-panel surface V/Vinf field, each
                                           panel framed by the actual solved
                                           (mirrored) geometry bounding box
                                           with a freestream direction arrow
  BWB_VSP_AERO_median_HV_with_CFD.png   - front + velocity field combined

plot_field(..., field="Cp", ...) still works if a pressure-coefficient
figure is wanted, it's just not called by main() by default.

Requires the `openvsp` package only for the (cached) solver runs - if the
.npz cache already exists for a design, plotting reuses it without invoking
VSPAERO again.

Run with: python plot_median_hv_bwb_cfd.py
"""

import contextlib
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().resolve().parent.parent.parent.parent
BWB_REPO = Path.cwd().resolve().parent
RESULTS_DIR = REPO_ROOT / "Results" / "BWB_VSP_AERO"
OUT_DIR = Path.cwd().resolve() / "bwb_cfd_median_hv"
OUT_DIR.mkdir(exist_ok=True)

DESIGN_KEYS = ["B1", "B2", "B3", "C2", "C3", "C4", "S1", "S2", "S3"]
FLIGHT_CONDITION = {"alt_kft": 35.0, "M_inf": 0.45, "Re": 4.0e7, "alpha_deg": 4.0}
GAMMA = 1.4

_PANELS = [("XSec_1", "B1", "S1", "C2"), ("XSec_2", "B2", "S2", "C3"), ("XSec_3", "B3", "S3", "C4")]


def median_hv_run_number() -> int:
    """Run number identified as the median-HV run for BWB_VSP_AERO."""
    with open(RESULTS_DIR / "Hypervolume" / "hypervolume_results.json") as f:
        hv = json.load(f)
    by_run = hv["hypervolume_by_run"]
    median = hv["statistics"]["median"]
    return int(min(by_run, key=lambda r: abs(by_run[r] - median)))


# def representative_designs(run_number: int):
#     """min-CD extreme, knee point, and max-CL extreme off the run's true ND front.

#     Returns (designs, idx): idx maps each label to its row in the front's F/X
#     arrays, so callers can plot the marker at the *exact* archived front
#     point rather than at a freshly re-solved (and therefore slightly
#     different, since the VLM re-solve here uses a different tessellation
#     than whatever produced the archive) CL/CD.
#     """
#     front_dir = RESULTS_DIR / "Robust_ND_front_points"
#     tag = f"run_{run_number:02d}"
#     X = np.load(front_dir / f"X_robust_nd_front_true_{tag}.npy")
#     F = np.load(front_dir / f"robust_nd_front_true_{tag}.npy")  # F[:,0]=CD, F[:,1]=-CL

#     ideal, nadir = F.min(axis=0), F.max(axis=0)
#     dist = np.linalg.norm((F - ideal) / (nadir - ideal), axis=1)

#     idx = {"min_CD": int(np.argmin(F[:, 0])), "max_CL": int(np.argmin(F[:, 1])), "knee": int(np.argmin(dist))}
#     designs = {label: dict(zip(DESIGN_KEYS, X[i])) for label, i in idx.items()}
#     return designs, idx

def representative_designs(run_number: int):
    """min-CD extreme, knee point, and max-CL extreme off the run's true ND front.

    Knee point follows the paper's hyperdistance definition (Zou et al. 2019,
    Sec. 3.3): build a hyperplane from the ideal and nadir points in
    normalized objective space, then pick the solution with the SHORTEST
    combined distance to that hyperplane (d1 = projection length along the
    ideal->nadir direction, d2 = perpendicular distance to the hyperplane).
    This is different from (and geometrically more correct than) simply
    picking the point closest to the ideal point.

    Returns (designs, idx): idx maps each label to its row in the front's F/X
    arrays, so callers can plot the marker at the *exact* archived front
    point rather than at a freshly re-solved (and therefore slightly
    different, since the VLM re-solve here uses a different tessellation
    than whatever produced the archive) CL/CD.
    """
    front_dir = RESULTS_DIR / "Robust_ND_front_points"
    tag = f"run_{run_number:02d}"
    X = np.load(front_dir / f"X_robust_nd_front_true_{tag}.npy")
    F = np.load(front_dir / f"robust_nd_front_true_{tag}.npy")  # F[:,0]=CD, F[:,1]=-CL

    ideal, nadir = F.min(axis=0), F.max(axis=0)
    F_norm = (F - ideal) / (nadir - ideal)  # ideal -> (0,...,0), nadir -> (1,...,1)

    normal = np.ones(F_norm.shape[1])       # nadir - ideal in normalized space
    normal_unit = normal / np.linalg.norm(normal)

    d1 = F_norm @ normal_unit                                # projection length onto ideal->nadir direction
    proj_points = np.outer(d1, normal_unit)
    d2 = np.linalg.norm(F_norm - proj_points, axis=1)        # perpendicular distance to the hyperplane

    hyperdistance = d1 + d2
    idx_knee = int(np.argmin(hyperdistance))                 # shortest hyperdistance = knee point

    idx = {"min_CD": int(np.argmin(F[:, 0])), "max_CL": int(np.argmin(F[:, 1])), "knee": idx_knee}
    designs = {label: dict(zip(DESIGN_KEYS, X[i])) for label, i in idx.items()}
    return designs, idx


def cp_to_velocity_ratio(cp: np.ndarray, mach: float, gamma: float = GAMMA) -> np.ndarray:
    """V/Vinf from surface Cp via the compressible (isentropic) Bernoulli relation.

    Cp = (2/(gamma*M^2)) * [(1 + (gamma-1)/2 * M^2 * (1-(V/Vinf)^2))^(gamma/(gamma-1)) - 1]
    solved for V/Vinf. VLM has no off-body velocity field to sample directly,
    so this is the standard way to turn a solved surface Cp into a velocity
    contour.
    """
    a = 1.0 + cp * gamma * mach ** 2 / 2.0
    a = np.clip(a, 1e-9, None)  # guard against Cp below the physical floor
    bracket = a ** ((gamma - 1.0) / gamma)
    v_ratio_sq = 1.0 - (bracket - 1.0) * 2.0 / ((gamma - 1.0) * mach ** 2)
    return np.sqrt(np.clip(v_ratio_sq, 0.0, None))


def _run_vspaero_design(label: str, design: dict, fc: dict, n_y_slices: int = 41) -> Path:
    """Full VSPAERO VLM solve + spanwise CpSlicer sweep for one design; caches to .npz."""
    import openvsp as vsp
    from vspaero_lowfidelity import VSP3_PATH

    devnull = open(os.devnull, "w")
    tmp = tempfile.mkdtemp()
    vsp3_copy = Path(tmp) / "model.vsp3"
    shutil.copyfile(VSP3_PATH, vsp3_copy)

    vsp.VSPCheckSetup()
    vsp.ClearVSPModel()
    vsp.ReadVSPFile(str(vsp3_copy))
    geom_id = vsp.FindGeomsWithName("WingGeom1")[0]
    vsp.SetParmVal(geom_id, "Tess_W", "Shape", 25.0)
    for xsec, b_key, s_key, tip_key in _PANELS:
        vsp.SetParmVal(geom_id, "Span", xsec, design[b_key])
        vsp.SetParmVal(geom_id, "Sweep", xsec, design[s_key])
        vsp.SetParmVal(geom_id, "Tip_Chord", xsec, design[tip_key])
    vsp.Update()

    vsp.SetAnalysisInputDefaults("VSPAEROComputeGeometry")
    vsp.SetIntAnalysisInput("VSPAEROComputeGeometry", "GeomSet", [-1])
    vsp.SetIntAnalysisInput("VSPAEROComputeGeometry", "ThinGeomSet", [1])
    vsp.Update()
    with contextlib.redirect_stdout(devnull):
        vsp.ExecAnalysis("VSPAEROComputeGeometry")

    analysis = "VSPAEROSweep"
    vsp.SetAnalysisInputDefaults(analysis)
    vsp.SetIntAnalysisInput(analysis, "GeomSet", [-1])
    vsp.SetIntAnalysisInput(analysis, "ThinGeomSet", [1])
    vsp.SetIntAnalysisInput(analysis, "RefFlag", [1])
    vsp.SetStringAnalysisInput(analysis, "WingID", [geom_id])
    vsp.SetDoubleAnalysisInput(analysis, "AlphaStart", [fc["alpha_deg"]])
    vsp.SetIntAnalysisInput(analysis, "AlphaNpts", [1])
    vsp.SetDoubleAnalysisInput(analysis, "MachStart", [fc["M_inf"]])
    vsp.SetIntAnalysisInput(analysis, "MachNpts", [1])
    vsp.SetDoubleAnalysisInput(analysis, "ReCref", [fc["Re"]])
    vsp.SetIntAnalysisInput(analysis, "NCPU", [4])
    vsp.Update()
    with contextlib.redirect_stdout(devnull):
        vsp.ExecAnalysis(analysis)

    hist_id = vsp.FindLatestResultsID("VSPAERO_History")
    cl = vsp.GetDoubleResults(hist_id, "CLtot")[-1]
    cd = vsp.GetDoubleResults(hist_id, "CDtot")[-1]
    cmy = vsp.GetDoubleResults(hist_id, "CMytot")[-1]

    ymax = 1086.93  # half-span extent of this model's Bref; slice just inboard of the tips
    # (CpSlicer degenerates exactly at the tip edge, so 0.995 rather than 1.0 - this
    # is close enough that the "missing" sliver at each tip is sub-pixel at plot scale)
    y_positions = np.linspace(-ymax * 0.995, ymax * 0.995, n_y_slices)
    vsp.SetAnalysisInputDefaults("CpSlicer")
    vsp.SetDoubleAnalysisInput("CpSlicer", "YSlicePosVec", list(y_positions))
    with contextlib.redirect_stdout(devnull):
        cp_res = vsp.ExecAnalysis("CpSlicer")
    case_ids = vsp.GetStringResults(cp_res, "CpSlice_Case_ID_Vec")

    X, Y, Z, CP = [], [], [], []
    for cid in case_ids:
        X.extend(vsp.GetDoubleResults(cid, "X_Loc"))
        Y.extend(vsp.GetDoubleResults(cid, "Y_Loc"))
        Z.extend(vsp.GetDoubleResults(cid, "Z_Loc"))
        CP.extend(vsp.GetDoubleResults(cid, "Cp"))

    out_path = OUT_DIR / f"bwb_{label}.npz"
    np.savez(out_path, X=np.array(X), Y=np.array(Y), Z=np.array(Z), CP=np.array(CP),
              CL=cl, CD=cd, CMy=cmy, design=json.dumps(design))
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"{label}: CL={cl:.5f} CD={cd:.5f} CMy={cmy:.5f} n_pts={len(X)}")
    return out_path


def solve_or_load(designs: dict, fc: dict) -> dict:
    data = {}
    for label, design in designs.items():
        cache = OUT_DIR / f"bwb_{label}.npz"
        if not cache.exists():
            sys.path.insert(0, str(BWB_REPO / "optimization"))
            _run_vspaero_design(label, design, fc)
        data[label] = np.load(cache)
    return data


def structured_chord_grid(d: dict, n_chord: int = 60):
    """Resample each spanwise Cp slice onto a common normalized-chord grid so the
    field can be drawn as a proper (span x chord) mesh instead of a naive
    Delaunay triangulation (which bridges across the taper/sweep and produces
    spurious spikes near the root)."""
    X, Y, CP = d["X"], d["Y"], d["CP"]
    rows = []
    for yv in np.unique(np.round(Y, 3)):
        mask = np.isclose(Y, yv, atol=1e-2)
        if mask.sum() < 3:
            continue
        xs, cps = X[mask], CP[mask]
        order = np.argsort(xs)
        xs, cps = xs[order], cps[order]
        xs, uidx = np.unique(xs, return_index=True)
        cps = cps[uidx]
        if len(xs) < 3:
            continue
        s = (xs - xs.min()) / (xs.max() - xs.min())
        s_grid = np.linspace(0, 1, n_chord)
        rows.append((yv, xs.min() + s_grid * (xs.max() - xs.min()), np.interp(s_grid, s, cps)))
    rows.sort(key=lambda r: r[0])
    Ymesh = np.repeat(np.array([r[0] for r in rows])[:, None], n_chord, axis=1)
    Xgrid = np.array([r[1] for r in rows])
    CPgrid = np.array([r[2] for r in rows])
    return Ymesh, Xgrid, CPgrid


LABELS = [("min_CD", "Min-Drag Extreme", "#2ca02c", "o"),
          ("knee", "Knee-Point", "#d62728", "D"),
          ("max_CL", "Max-Lift Extreme", "#9467bd", "s")]


def add_geometry_frame(ax, d: dict):
    """Draws the actual solved (mirrored) geometry bounding box around the
    field plot, plus a freestream direction arrow.

    VSPAERO itself computes an exact Xmin/Xmax, Ymin/Ymax, Zmin/Zmax bounding
    box of the geometry internally (visible in its solver log) to size the
    wake/far-field domain - it isn't rendered anywhere in the model or GUI,
    so nothing shows it by default. Here it's taken directly from the extents
    of the solved CpSlice points (X, Y already cover the true, symmetry-
    mirrored surface), not an idealized/parametric approximation.
    """
    x_min, x_max = float(d["X"].min()), float(d["X"].max())
    y_min, y_max = float(d["Y"].min()), float(d["Y"].max())
    x_span, y_span = x_max - x_min, y_max - y_min

    rect = plt.Rectangle((y_min, x_min), y_span, x_span,
                          fill=False, edgecolor="0.25", linewidth=1.1,
                          linestyle="--", alpha=0.8, zorder=6)
    ax.add_patch(rect)

    # Freestream arrow: flow runs root-LE (small X) -> TE (large X); the axis
    # is inverted so that direction reads as pointing down the page.
    arrow_x = 0.0
    x_tail, x_head = x_min - 0.22 * x_span, x_min - 0.04 * x_span
    ax.annotate("", xy=(arrow_x, x_head), xytext=(arrow_x, x_tail),
                arrowprops=dict(arrowstyle="-|>", color="0.15", lw=1.6), zorder=7)
    ax.text(arrow_x, x_tail - 0.05 * x_span, r"$V_\infty$", ha="center", va="top",
            fontsize=9.5, color="0.15")

    # Margin on all 4 sides so the box doesn't hide behind the axes spines.
    ax.set_xlim(y_min - 0.06 * y_span, y_max + 0.06 * y_span)
    ax.set_ylim(x_max + 0.05 * x_span, x_min - 0.32 * x_span)


def plot_front(run_number: int, idx: dict, out_path: Path):
    front_dir = RESULTS_DIR / "Robust_ND_front_points"
    F = np.load(front_dir / f"robust_nd_front_true_run_{run_number:02d}.npy")
    CD, CL_obj = F[:, 0], F[:, 1]  # CL kept negative, matching the optimizer's own objective (F[:,1] = -CL)

    fig, ax = plt.subplots(figsize=(6.2, 5))
    ax.scatter(CD, CL_obj, s=22, c="#4C72B0", alpha=0.55, edgecolor="none",
               label=f"Run {run_number} (median-HV) ND front")
    for label, title, color, marker in LABELS:
        # Plot the archived front point itself (not the freshly re-solved CL/CD from
        # `data`, which uses a different VLM tessellation and so lands slightly off
        # the front curve) - this guarantees the marker sits exactly on the front.
        cd, cl_obj = float(CD[idx[label]]), float(CL_obj[idx[label]])
        ax.scatter(cd, cl_obj, s=140, c=color, marker=marker, edgecolor="black", linewidth=1.1, zorder=5, label=title)
        ax.annotate(title, (cd, cl_obj), textcoords="offset points", xytext=(8, 6),
                    fontsize=10, fontweight="bold", color=color)
    ax.set_xlabel("$C_D$"); ax.set_ylabel("$-C_L$")
    ax.set_title(f"BWB VSP-Aero - Median-HV Run (Run {run_number}) Robust ND Front")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=9)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)


def plot_field(data: dict, field: str, cbar_label: str, cmap: str, out_path: Path, fc: dict, title_suffix: str = ""):
    grids, values = {}, {}
    for label, _, _, _ in LABELS:
        Ymesh, Xg, CPg = structured_chord_grid(data[label])
        grids[label] = (Ymesh, Xg)
        if field == "Cp":
            values[label] = CPg
        else:
            values[label] = cp_to_velocity_ratio(CPg, fc["M_inf"])

    all_vals = np.concatenate([v.ravel() for v in values.values()])
    if field == "Cp":
        vabs = np.percentile(np.abs(all_vals), 98)
        vmin, vmax = -vabs, vabs
    else:
        vmin, vmax = np.percentile(all_vals, [1, 99])

    fig, axes = plt.subplots(1, 3, figsize=(16, 6.2), sharey=True)
    for ax, (label, title, color, _) in zip(axes, LABELS):
        Ymesh, Xg = grids[label]
        d = data[label]
        pc = ax.pcolormesh(Ymesh, Xg, values[label], shading="gouraud", cmap=cmap, vmin=vmin, vmax=vmax)
        ax.contour(Ymesh, Xg, values[label], levels=14, colors="k", linewidths=0.25, alpha=0.35)
        add_geometry_frame(ax, d)
        # ax.set_title(f"{title}\nCL={float(d['CL']):.3f}  CD={float(d['CD']):.4f}  CMy={float(d['CMy']):.3f}",
        #              fontsize=10, color=color)
        ax.set_xlabel("Y (mm, span)")
        ax.set_aspect("equal")
    axes[0].set_ylabel("X (mm, chord, LE at top)")
    cbar = fig.colorbar(pc, ax=axes, shrink=0.45, pad=0.02)
    cbar.set_label(cbar_label)
    # fig.suptitle(f"BWB VSP-Aero - Median-HV Run: {title_suffix}"
    #              f" alpha={fc['alpha_deg']:.0f}deg, M={fc['M_inf']}", fontsize=13)
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def plot_combined(run_number: int, data: dict, idx: dict, fc: dict, out_path: Path):
    front_dir = RESULTS_DIR / "Robust_ND_front_points"
    F = np.load(front_dir / f"robust_nd_front_true_run_{run_number:02d}.npy")
    CD, CL_obj = F[:, 0], F[:, 1]  # CL kept negative, matching the optimizer's own objective (F[:,1] = -CL)

    grids, v_values = {}, {}
    for label, _, _, _ in LABELS:
        Ymesh, Xg, CPg = structured_chord_grid(data[label])
        grids[label] = (Ymesh, Xg)
        v_values[label] = cp_to_velocity_ratio(CPg, fc["M_inf"])
    vmin, vmax = np.percentile(np.concatenate([v.ravel() for v in v_values.values()]), [1, 99])

    fig = plt.figure(figsize=(15, 10))
    gs = gridspec.GridSpec(2, 3, height_ratios=[1.05, 1], hspace=0.42, wspace=0.25)

    ax_front = fig.add_subplot(gs[0, :])
    ax_front.scatter(CD, CL_obj, s=22, c="#4C72B0", alpha=0.55, edgecolor="none",
                      label=f"Run {run_number} (median-HV) ND front")
    for label, title, color, marker in LABELS:
        # Archived front point (see plot_front) so the marker sits exactly on the front.
        cd, cl_obj = float(CD[idx[label]]), float(CL_obj[idx[label]])
        ax_front.scatter(cd, cl_obj, s=140, c=color, marker=marker, edgecolor="black", linewidth=1.1, zorder=5, label=title)
        ax_front.annotate(title, (cd, cl_obj), textcoords="offset points", xytext=(8, 6),
                           fontsize=9.5, fontweight="bold", color=color)
    ax_front.set_xlabel("$C_D$"); ax_front.set_ylabel("$-C_L$")
    ax_front.set_title(f"BWB VSP-Aero -- Median-HV Run (Run {run_number}) Robust Non-Dominated Front")
    ax_front.grid(alpha=0.3)
    ax_front.legend(loc="upper right", fontsize=8.5)

    axes_v = [fig.add_subplot(gs[1, i]) for i in range(3)]
    for ax, (label, title, color, _) in zip(axes_v, LABELS):
        Ymesh, Xg = grids[label]
        d = data[label]
        pc = ax.pcolormesh(Ymesh, Xg, v_values[label], shading="gouraud", cmap="viridis", vmin=vmin, vmax=vmax)
        ax.contour(Ymesh, Xg, v_values[label], levels=14, colors="k", linewidths=0.25, alpha=0.35)
        add_geometry_frame(ax, d)
        ax.set_title(f"{title}\nCL={float(d['CL']):.3f}  CD={float(d['CD']):.4f}  CMy={float(d['CMy']):.3f}",
                     fontsize=10, color=color)
        ax.set_xlabel("Y (mm, span)")
        ax.set_aspect("equal")
    axes_v[0].set_ylabel("X (mm, chord, LE at top)")
    cbar = fig.colorbar(pc, ax=axes_v, shrink=0.85, pad=0.02)
    cbar.set_label("Local surface velocity ratio, $V/V_\\infty$")

    fig.suptitle(f"BWB VSP-Aero Median-HV Run: Robust ND Front & Surface Velocity (VLM), "
                 f"alpha={fc['alpha_deg']:.0f}deg, M={fc['M_inf']}", fontsize=13, y=0.98)
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def main():
    run_number = 20 #median_hv_run_number()
    print(f"Median-HV run: {run_number}")

    designs, idx = representative_designs(run_number)
    data = solve_or_load(designs, FLIGHT_CONDITION)

    plot_front(run_number, idx, OUT_DIR / f"BWB_run{run_number:02d}_front_highlighted.png")
    plot_field(data, "V", "Local surface velocity ratio, $V/V_\\infty$", "viridis",
               OUT_DIR / f"BWB_run{run_number:02d}_velocity_ratio.png", FLIGHT_CONDITION,
               title_suffix="Surface Velocity Ratio (from Cp via isentropic relation),")
    plot_combined(run_number, data, idx, FLIGHT_CONDITION, OUT_DIR / "BWB_VSP_AERO_median_HV_with_CFD.png")

    print(f"Figures written to {OUT_DIR}")


if __name__ == "__main__":
    main()


Median-HV run: 20
min_CD: CL=0.20719 CD=0.00719 CMy=-0.23499 n_pts=885
max_CL: CL=0.29426 CD=0.01115 CMy=-0.24271 n_pts=971
knee: CL=0.25625 CD=0.00902 CMy=-0.27045 n_pts=967
Figures written to C:\Users\z5653370\OneDrive - UNSW\Documents\PhD\JMD2\Problems\Constrained\BWB_VSP_Aero\optimization\bwb_cfd_median_hv


In [16]:

import contextlib
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent.parent.parent.parent
BWB_REPO = Path.cwd().resolve().parent
RESULTS_DIR = REPO_ROOT / "Results" / "BWB_VSP_AERO"
OUT_DIR = Path.cwd().resolve() / "bwb_cfd_median_hv"
OUT_DIR.mkdir(exist_ok=True)

DESIGN_KEYS = ["B1", "B2", "B3", "C2", "C3", "C4", "S1", "S2", "S3"]
FLIGHT_CONDITION = {"alt_kft": 35.0, "M_inf": 0.45, "Re": 4.0e7, "alpha_deg": 4.0}
GAMMA = 1.4
# Matches bwb_vsp_aero_problem.py's TESS_W default (the pipeline that actually
# produced the archived fronts) - keep these in sync, or CD comes out a
# consistent few % high/low relative to the archive (CL/CMy barely move,
# CD is what's sensitive to spanwise panel count).
TESS_W = 20.0

_PANELS = [("XSec_1", "B1", "S1", "C2"), ("XSec_2", "B2", "S2", "C3"), ("XSec_3", "B3", "S3", "C4")]


def median_hv_run_number() -> int:
    """Run number identified as the median-HV run for BWB_VSP_AERO.

    Results/Statistics/BWB_VSP_AERO_HV_statistics.json is the authoritative
    source - it aggregates all 31 runs and states "median_run" directly.
    (Results/BWB_VSP_AERO/Hypervolume/hypervolume_results.json looks similar
    but only has one run ("17") recorded under a different ideal/nadir - a
    stale/partial file, not the full-cohort statistics - don't use it here.)
    """
    with open(REPO_ROOT / "Results" / "Statistics" / "BWB_VSP_AERO_HV_statistics.json") as f:
        stats = json.load(f)
    return int(stats["statistics"]["median_run"])


def representative_designs(run_number: int):
    """min-CD extreme, knee point, and max-CL extreme off the run's true ND front.

    Returns (designs, idx): idx maps each label to its row in the front's F/X
    arrays, so callers can plot the marker at the *exact* archived front
    point rather than at a freshly re-solved (and therefore slightly
    different, since the VLM re-solve here uses a different tessellation
    than whatever produced the archive) CL/CD.
    """
    front_dir = RESULTS_DIR / "Robust_ND_front_points"
    tag = f"run_{run_number:02d}"
    X = np.load(front_dir / f"X_robust_nd_front_true_{tag}.npy")
    F = np.load(front_dir / f"robust_nd_front_true_{tag}.npy")  # F[:,0]=CD, F[:,1]=-CL

    ideal, nadir = F.min(axis=0), F.max(axis=0)
    dist = np.linalg.norm((F - ideal) / (nadir - ideal), axis=1)

    idx = {"min_CD": int(np.argmin(F[:, 0])), "max_CL": int(np.argmin(F[:, 1])), "knee": int(np.argmin(dist))}
    designs = {label: dict(zip(DESIGN_KEYS, X[i])) for label, i in idx.items()}
    return designs, idx


def cp_to_velocity_ratio(cp: np.ndarray, mach: float, gamma: float = GAMMA) -> np.ndarray:
    """V/Vinf from surface Cp via the compressible (isentropic) Bernoulli relation.

    Cp = (2/(gamma*M^2)) * [(1 + (gamma-1)/2 * M^2 * (1-(V/Vinf)^2))^(gamma/(gamma-1)) - 1]
    solved for V/Vinf. VLM has no off-body velocity field to sample directly,
    so this is the standard way to turn a solved surface Cp into a velocity
    contour.
    """
    a = 1.0 + cp * gamma * mach ** 2 / 2.0
    a = np.clip(a, 1e-9, None)  # guard against Cp below the physical floor
    bracket = a ** ((gamma - 1.0) / gamma)
    v_ratio_sq = 1.0 - (bracket - 1.0) * 2.0 / ((gamma - 1.0) * mach ** 2)
    return np.sqrt(np.clip(v_ratio_sq, 0.0, None))


def _run_vspaero_design(label: str, design: dict, fc: dict, run_number: int, tess_w: float = TESS_W,
                         n_y_slices: int = 41) -> Path:
    """Full VSPAERO VLM solve + spanwise CpSlicer sweep for one design; caches to .npz."""
    import openvsp as vsp
    from vspaero_lowfidelity import VSP3_PATH

    devnull = open(os.devnull, "w")
    tmp = tempfile.mkdtemp()
    vsp3_copy = Path(tmp) / "model.vsp3"
    shutil.copyfile(VSP3_PATH, vsp3_copy)

    vsp.VSPCheckSetup()
    vsp.ClearVSPModel()
    vsp.ReadVSPFile(str(vsp3_copy))
    geom_id = vsp.FindGeomsWithName("WingGeom1")[0]
    vsp.SetParmVal(geom_id, "Tess_W", "Shape", float(tess_w))
    for xsec, b_key, s_key, tip_key in _PANELS:
        vsp.SetParmVal(geom_id, "Span", xsec, design[b_key])
        vsp.SetParmVal(geom_id, "Sweep", xsec, design[s_key])
        vsp.SetParmVal(geom_id, "Tip_Chord", xsec, design[tip_key])
    vsp.Update()

    vsp.SetAnalysisInputDefaults("VSPAEROComputeGeometry")
    vsp.SetIntAnalysisInput("VSPAEROComputeGeometry", "GeomSet", [-1])
    vsp.SetIntAnalysisInput("VSPAEROComputeGeometry", "ThinGeomSet", [1])
    vsp.Update()
    with contextlib.redirect_stdout(devnull):
        vsp.ExecAnalysis("VSPAEROComputeGeometry")

    analysis = "VSPAEROSweep"
    vsp.SetAnalysisInputDefaults(analysis)
    vsp.SetIntAnalysisInput(analysis, "GeomSet", [-1])
    vsp.SetIntAnalysisInput(analysis, "ThinGeomSet", [1])
    vsp.SetIntAnalysisInput(analysis, "RefFlag", [1])
    vsp.SetStringAnalysisInput(analysis, "WingID", [geom_id])
    vsp.SetDoubleAnalysisInput(analysis, "AlphaStart", [fc["alpha_deg"]])
    vsp.SetIntAnalysisInput(analysis, "AlphaNpts", [1])
    vsp.SetDoubleAnalysisInput(analysis, "MachStart", [fc["M_inf"]])
    vsp.SetIntAnalysisInput(analysis, "MachNpts", [1])
    vsp.SetDoubleAnalysisInput(analysis, "ReCref", [fc["Re"]])
    vsp.SetIntAnalysisInput(analysis, "NCPU", [4])
    vsp.Update()
    with contextlib.redirect_stdout(devnull):
        vsp.ExecAnalysis(analysis)

    hist_id = vsp.FindLatestResultsID("VSPAERO_History")
    cl = vsp.GetDoubleResults(hist_id, "CLtot")[-1]
    cd = vsp.GetDoubleResults(hist_id, "CDtot")[-1]
    cmy = vsp.GetDoubleResults(hist_id, "CMytot")[-1]

    ymax = 1086.93  # half-span extent of this model's Bref; slice just inboard of the tips
    # (CpSlicer degenerates exactly at the tip edge, so 0.995 rather than 1.0 - this
    # is close enough that the "missing" sliver at each tip is sub-pixel at plot scale)
    y_positions = np.linspace(-ymax * 0.995, ymax * 0.995, n_y_slices)
    vsp.SetAnalysisInputDefaults("CpSlicer")
    vsp.SetDoubleAnalysisInput("CpSlicer", "YSlicePosVec", list(y_positions))
    with contextlib.redirect_stdout(devnull):
        cp_res = vsp.ExecAnalysis("CpSlicer")
    case_ids = vsp.GetStringResults(cp_res, "CpSlice_Case_ID_Vec")

    X, Y, Z, CP = [], [], [], []
    for cid in case_ids:
        X.extend(vsp.GetDoubleResults(cid, "X_Loc"))
        Y.extend(vsp.GetDoubleResults(cid, "Y_Loc"))
        Z.extend(vsp.GetDoubleResults(cid, "Z_Loc"))
        CP.extend(vsp.GetDoubleResults(cid, "Cp"))

    out_path = OUT_DIR / f"bwb_run{run_number:02d}_tess{int(tess_w)}_{label}.npz"
    np.savez(out_path, X=np.array(X), Y=np.array(Y), Z=np.array(Z), CP=np.array(CP),
              CL=cl, CD=cd, CMy=cmy, design=json.dumps(design))
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"{label}: CL={cl:.5f} CD={cd:.5f} CMy={cmy:.5f} n_pts={len(X)}")
    return out_path


def solve_or_load(designs: dict, fc: dict, run_number: int, tess_w: float = TESS_W) -> dict:
    """Cache filenames are keyed by run_number and tess_w as well as label -
    otherwise switching which run is "median" (e.g. after fixing
    median_hv_run_number()) or changing the mesh resolution would silently
    reuse stale cached aero data under labels that now mean something else."""
    data = {}
    for label, design in designs.items():
        cache = OUT_DIR / f"bwb_run{run_number:02d}_tess{int(tess_w)}_{label}.npz"
        if not cache.exists():
            sys.path.insert(0, str(BWB_REPO / "optimization"))
            _run_vspaero_design(label, design, fc, run_number, tess_w)
        data[label] = np.load(cache)
    return data


def structured_chord_grid(d: dict, n_chord: int = 60):
    """Resample each spanwise Cp slice onto a common normalized-chord grid so the
    field can be drawn as a proper (span x chord) mesh instead of a naive
    Delaunay triangulation (which bridges across the taper/sweep and produces
    spurious spikes near the root)."""
    X, Y, CP = d["X"], d["Y"], d["CP"]
    rows = []
    for yv in np.unique(np.round(Y, 3)):
        mask = np.isclose(Y, yv, atol=1e-2)
        if mask.sum() < 3:
            continue
        xs, cps = X[mask], CP[mask]
        order = np.argsort(xs)
        xs, cps = xs[order], cps[order]
        xs, uidx = np.unique(xs, return_index=True)
        cps = cps[uidx]
        if len(xs) < 3:
            continue
        s = (xs - xs.min()) / (xs.max() - xs.min())
        s_grid = np.linspace(0, 1, n_chord)
        rows.append((yv, xs.min() + s_grid * (xs.max() - xs.min()), np.interp(s_grid, s, cps)))
    rows.sort(key=lambda r: r[0])
    Ymesh = np.repeat(np.array([r[0] for r in rows])[:, None], n_chord, axis=1)
    Xgrid = np.array([r[1] for r in rows])
    CPgrid = np.array([r[2] for r in rows])
    return Ymesh, Xgrid, CPgrid


LABELS = [("min_CD", "Min-Drag Extreme", "#2ca02c", "o"),
          ("knee", "Knee-Point", "#d62728", "D"),
          ("max_CL", "Max-Lift Extreme", "#9467bd", "s")]


def add_geometry_frame(ax, d: dict):
    """Draws the actual solved (mirrored) geometry bounding box around the
    field plot, plus a freestream direction arrow.

    VSPAERO itself computes an exact Xmin/Xmax, Ymin/Ymax, Zmin/Zmax bounding
    box of the geometry internally (visible in its solver log) to size the
    wake/far-field domain - it isn't rendered anywhere in the model or GUI,
    so nothing shows it by default. Here it's taken directly from the extents
    of the solved CpSlice points (X, Y already cover the true, symmetry-
    mirrored surface), not an idealized/parametric approximation.
    """
    x_min, x_max = float(d["X"].min()), float(d["X"].max())
    y_min, y_max = float(d["Y"].min()), float(d["Y"].max())
    x_span, y_span = x_max - x_min, y_max - y_min

    rect = plt.Rectangle((y_min, x_min), y_span, x_span,
                          fill=False, edgecolor="0.25", linewidth=1.1,
                          linestyle="--", alpha=0.8, zorder=6)
    ax.add_patch(rect)

    # Freestream arrow: flow runs root-LE (small X) -> TE (large X); the axis
    # is inverted so that direction reads as pointing down the page.
    arrow_x = 0.0
    x_tail, x_head = x_min - 0.22 * x_span, x_min - 0.04 * x_span
    ax.annotate("", xy=(arrow_x, x_head), xytext=(arrow_x, x_tail),
                arrowprops=dict(arrowstyle="-|>", color="0.15", lw=1.6), zorder=7)
    ax.text(arrow_x, x_tail - 0.05 * x_span, r"$V_\infty$", ha="center", va="top",
            fontsize=9.5, color="0.15")

    # Margin on all 4 sides so the box doesn't hide behind the axes spines.
    ax.set_xlim(y_min - 0.06 * y_span, y_max + 0.06 * y_span)
    ax.set_ylim(x_max + 0.05 * x_span, x_min - 0.32 * x_span)


def plot_front(run_number: int, idx: dict, out_path: Path):
    front_dir = RESULTS_DIR / "Robust_ND_front_points"
    F = np.load(front_dir / f"robust_nd_front_true_run_{run_number:02d}.npy")
    CD, CL_obj = F[:, 0], F[:, 1]  # CL kept negative, matching the optimizer's own objective (F[:,1] = -CL)

    fig, ax = plt.subplots(figsize=(6.2, 5))
    ax.scatter(CD, CL_obj, s=22, c="#4C72B0", alpha=0.55, edgecolor="none",
               label=f"Run {run_number} (median-HV) ND front")
    for label, title, color, marker in LABELS:
        # Plot the archived front point itself (not the freshly re-solved CL/CD from
        # `data`, which uses a different VLM tessellation and so lands slightly off
        # the front curve) - this guarantees the marker sits exactly on the front.
        cd, cl_obj = float(CD[idx[label]]), float(CL_obj[idx[label]])
        ax.scatter(cd, cl_obj, s=140, c=color, marker=marker, edgecolor="black", linewidth=1.1, zorder=5, label=title)
        ax.annotate(title, (cd, cl_obj), textcoords="offset points", xytext=(8, 6),
                    fontsize=10, fontweight="bold", color=color)
    ax.set_xlabel("$C_D$"); ax.set_ylabel("$-C_L$")
    ax.set_title(f"BWB VSP-Aero - Median-HV Run (Run {run_number}) Robust ND Front")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=9)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)


def plot_field(data: dict, field: str, cbar_label: str, cmap: str, out_path: Path, fc: dict, title_suffix: str = ""):
    grids, values = {}, {}
    for label, _, _, _ in LABELS:
        Ymesh, Xg, CPg = structured_chord_grid(data[label])
        grids[label] = (Ymesh, Xg)
        if field == "Cp":
            values[label] = CPg
        else:
            values[label] = cp_to_velocity_ratio(CPg, fc["M_inf"])

    all_vals = np.concatenate([v.ravel() for v in values.values()])
    if field == "Cp":
        vabs = np.percentile(np.abs(all_vals), 98)
        vmin, vmax = -vabs, vabs
    else:
        vmin, vmax = np.percentile(all_vals, [1, 99])

    fig, axes = plt.subplots(1, 3, figsize=(16, 6.2), sharey=True)
    for ax, (label, title, color, _) in zip(axes, LABELS):
        Ymesh, Xg = grids[label]
        d = data[label]
        pc = ax.pcolormesh(Ymesh, Xg, values[label], shading="gouraud", cmap=cmap, vmin=vmin, vmax=vmax)
        ax.contour(Ymesh, Xg, values[label], levels=14, colors="k", linewidths=0.25, alpha=0.35)
        add_geometry_frame(ax, d)
        # ax.set_title(f"{title}\nCL={float(d['CL']):.3f}  CD={float(d['CD']):.4f}  CMy={float(d['CMy']):.3f}",
        #              fontsize=10, color=color)
        ax.set_xlabel("Y (mm, span)")
        ax.set_aspect("equal")
    axes[0].set_ylabel("X (mm, chord, LE at top)")
    cbar = fig.colorbar(pc, ax=axes, shrink=0.45, pad=0.02)
    cbar.set_label(cbar_label)
    # fig.suptitle(f"BWB VSP-Aero - Median-HV Run: {title_suffix}"
    #              f" alpha={fc['alpha_deg']:.0f}deg, M={fc['M_inf']}", fontsize=13)
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def plot_combined(run_number: int, data: dict, idx: dict, fc: dict, out_path: Path):
    front_dir = RESULTS_DIR / "Robust_ND_front_points"
    F = np.load(front_dir / f"robust_nd_front_true_run_{run_number:02d}.npy")
    CD, CL_obj = F[:, 0], F[:, 1]  # CL kept negative, matching the optimizer's own objective (F[:,1] = -CL)

    grids, v_values = {}, {}
    for label, _, _, _ in LABELS:
        Ymesh, Xg, CPg = structured_chord_grid(data[label])
        grids[label] = (Ymesh, Xg)
        v_values[label] = cp_to_velocity_ratio(CPg, fc["M_inf"])
    vmin, vmax = np.percentile(np.concatenate([v.ravel() for v in v_values.values()]), [1, 99])

    fig = plt.figure(figsize=(15, 10))
    gs = gridspec.GridSpec(2, 3, height_ratios=[1.05, 1], hspace=0.42, wspace=0.25)

    ax_front = fig.add_subplot(gs[0, :])
    ax_front.scatter(CD, CL_obj, s=22, c="#4C72B0", alpha=0.55, edgecolor="none",
                      label=f"Run {run_number} (median-HV) ND front")
    for label, title, color, marker in LABELS:
        # Archived front point (see plot_front) so the marker sits exactly on the front.
        cd, cl_obj = float(CD[idx[label]]), float(CL_obj[idx[label]])
        ax_front.scatter(cd, cl_obj, s=140, c=color, marker=marker, edgecolor="black", linewidth=1.1, zorder=5, label=title)
        ax_front.annotate(title, (cd, cl_obj), textcoords="offset points", xytext=(8, 6),
                           fontsize=9.5, fontweight="bold", color=color)
    ax_front.set_xlabel("$C_D$"); ax_front.set_ylabel("$-C_L$")
    ax_front.set_title(f"BWB VSP-Aero -- Median-HV Run (Run {run_number}) Robust Non-Dominated Front")
    ax_front.grid(alpha=0.3)
    ax_front.legend(loc="upper right", fontsize=8.5)

    axes_v = [fig.add_subplot(gs[1, i]) for i in range(3)]
    for ax, (label, title, color, _) in zip(axes_v, LABELS):
        Ymesh, Xg = grids[label]
        d = data[label]
        pc = ax.pcolormesh(Ymesh, Xg, v_values[label], shading="gouraud", cmap="viridis", vmin=vmin, vmax=vmax)
        ax.contour(Ymesh, Xg, v_values[label], levels=14, colors="k", linewidths=0.25, alpha=0.35)
        add_geometry_frame(ax, d)
        # ax.set_title(f"{title}\nCL={float(d['CL']):.3f}  CD={float(d['CD']):.4f}  CMy={float(d['CMy']):.3f}",
        #              fontsize=10, color=color)
        ax.set_xlabel("Y (mm, span)")
        ax.set_aspect("equal")
    axes_v[0].set_ylabel("X (mm, chord, LE at top)")
    cbar = fig.colorbar(pc, ax=axes_v, shrink=0.45, pad=0.02)
    cbar.set_label("Local surface velocity ratio, $V/V_\\infty$")

    # fig.suptitle(f"BWB VSP-Aero Median-HV Run: Robust ND Front & Surface Velocity (VLM), "
    #              f"alpha={fc['alpha_deg']:.0f}deg, M={fc['M_inf']}", fontsize=13, y=0.98)
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def run_and_plot(run_number: int | None = None, tess_w: float = TESS_W, fc: dict = FLIGHT_CONDITION) -> dict:
    """Entry point: feed a run number (or None for the current median-HV run)
    and this solves + plots the min-CD/knee/max-CL designs off that run's
    true ND front.

    Returns {"run_number", "data", "idx"} in case you want to inspect the
    solved CL/CD/CMy or the front-index mapping afterward.

    Examples:
        run_and_plot()      # auto: whichever run is median-HV right now
        run_and_plot(20)    # a specific run, e.g. to compare against the median
    """
    if run_number is None:
        run_number = median_hv_run_number()
    print(f"Run: {run_number}  (Tess_W={tess_w})")

    designs, idx = representative_designs(run_number)
    data = solve_or_load(designs, fc, run_number, tess_w)

    plot_front(run_number, idx, OUT_DIR / f"BWB_run{run_number:02d}_front_highlighted.png")
    plot_field(data, "V", "Local surface velocity ratio, $V/V_\\infty$", "viridis",
               OUT_DIR / f"BWB_run{run_number:02d}_velocity_ratio.png", fc,
               title_suffix="Surface Velocity Ratio (from Cp via isentropic relation),")
    plot_combined(run_number, data, idx, fc, OUT_DIR / f"BWB_run{run_number:02d}_median_HV_with_CFD.png")

    print(f"Figures written to {OUT_DIR}")
    return {"run_number": run_number, "data": data, "idx": idx}


def main():
    median_run = 20
    tess_w = 20
    run_and_plot(median_run, tess_w)


if __name__ == "__main__":
    main()


Run: 20  (Tess_W=20)
Figures written to C:\Users\z5653370\OneDrive - UNSW\Documents\PhD\JMD2\Problems\Constrained\BWB_VSP_Aero\optimization\bwb_cfd_median_hv


In [75]:
"""Illustrates the 95th-percentile dominance selection (Sec. III of the paper)
using the actual SSARO selection code, not a synthetic approximation.

Produces three panels:
  (a) normal case: 100 points sampled uniformly in [0,1]^2, alpha=0.95
  (b) tie-break case: a sample where 2-3 points land equally close to the
      95th-percentile domination count, so the random tie-break fires
  (c) degenerate case: points placed exactly on a Pareto front, so every
      domination count is 0 and every point ties for the 95th percentile
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ==========================================================
# --- verbatim from the SSARO codebase ---
# ==========================================================

def domination_count(F):
    """
    Exact dominator count for 2-objective minimization.
    O(N log N), much faster than pairwise O(N^2). Fenwick approach
    """
    F = np.asarray(F, dtype=float)
    n = F.shape[0]

    f2_vals, f2_rank = np.unique(F[:, 1], return_inverse=True)
    tree = np.zeros(len(f2_vals) + 1, dtype=int)

    def add(i, v=1):
        i += 1
        while i < len(tree):
            tree[i] += v
            i += i & -i

    def sum_leq(i):
        i += 1
        s = 0
        while i > 0:
            s += tree[i]
            i -= i & -i
        return s

    counts = np.zeros(n, dtype=int)
    order = np.lexsort((F[:, 1], F[:, 0]))

    start = 0
    while start < n:
        end = start
        f1 = F[order[start], 0]

        while end < n and F[order[end], 0] == f1:
            end += 1

        group = order[start:end]
        group_sorted = group[np.argsort(F[group, 1])]

        # prior groups: f1 smaller, f2 <= current
        for idx in group_sorted:
            counts[idx] += sum_leq(f2_rank[idx])

        # same f1 group: only smaller f2 dominates
        _, first_pos = np.unique(F[group_sorted, 1], return_index=True)
        for pos, idx in enumerate(group_sorted):
            counts[idx] += first_pos[np.searchsorted(F[group_sorted[first_pos], 1], F[idx, 1])]

        for idx in group:
            add(f2_rank[idx])

        start = end

    return counts


def select_alpha_dominated_feasible_or_least_infeasible(F, G, alpha=0.95, seed=None):
    """
    Select an alpha-percentile dominated objective sample.

    Tie-breaking:
        If multiple solutions are equally close to the alpha-percentile
        domination count, one is selected uniformly at random.

    Handles:
        1. Unconstrained problems
        2. Constrained problems with feasible samples
        3. Constrained problems with no feasible samples
    """

    rng = np.random.default_rng(seed)

    F = np.asarray(F, dtype=float)
    G = np.asarray(G, dtype=float)

    def select_by_alpha_random_tie(F_sub, counts, alpha):
        counts = np.asarray(counts, dtype=float)

        q = np.quantile(counts, alpha)
        dist = np.abs(counts - q)

        candidate_idx = np.where(dist == dist.min())[0]

        return rng.choice(candidate_idx)

    # Unconstrained problem
    if G.size == 0:
        if alpha == "mean":
            return np.mean(F, axis=0)

        counts = domination_count(F)

        local_idx = select_by_alpha_random_tie(F, counts, alpha)
        return F[local_idx]

    if G.ndim == 1:
        G = G.reshape(-1, 1)

    feasible_mask = np.all(G <= 0.0, axis=1)

    if np.any(feasible_mask):
        feasible_indices = np.where(feasible_mask)[0]
        F_feas = F[feasible_mask]
        M = F_feas.shape[0]

        if alpha == "mean":
            return np.mean(F_feas, axis=0)

        if M == 1:
            return F_feas[0]

        counts = domination_count(F_feas)

        local_idx = select_by_alpha_random_tie(F_feas, counts, alpha)
        return F[feasible_indices[local_idx]]

    CV = np.sum(np.maximum(G, 0.0), axis=1)
    min_cv = np.min(CV)
    candidate_idx = np.where(CV == min_cv)[0]
    least_infeasible_idx = rng.choice(candidate_idx)
    return F[least_infeasible_idx]


def process_design(F_samples_i, G_samples_i, alpha):
    if alpha == "mean":
        return np.mean(F_samples_i, axis=0)
    return select_alpha_dominated_feasible_or_least_infeasible(
        F_samples_i, G_samples_i, alpha=alpha,
    )


# ==========================================================
# --- plotting helpers (styling: large ticks, no title) ---
# ==========================================================

ALPHA = 0.95
TICK_FS, LABEL_FS, LEGEND_FS = 13, 14, 11
OUT = r"c:\Users\z5653370\OneDrive - UNSW\Documents\PhD\JMD2\Manuscript_Overleaf\Figures"


def plot_panel(F, counts, tie_idx, sel_idx, out_path, vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    sc = ax.scatter(F[:, 0], F[:, 1], c=counts, cmap="viridis", s=55,
                     edgecolor="black", linewidth=0.4, zorder=3, vmin=vmin, vmax=vmax)
    if tie_idx is not None and len(tie_idx) > 1:
        ax.scatter(F[tie_idx, 0], F[tie_idx, 1], s=150, facecolor="none",
                   edgecolor="crimson", linewidth=1.6, zorder=4)
    ax.scatter(F[sel_idx, 0], F[sel_idx, 1], marker="*", s=260, c="red",
               edgecolor="black", linewidth=0.8, zorder=5, label="Selected solution")
    ax.set_xlabel("$f_1$", fontsize=LABEL_FS)
    ax.set_ylabel("$f_2$", fontsize=LABEL_FS)
    ax.tick_params(axis="both", labelsize=TICK_FS)
    ax.grid(alpha=0.3)
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Domination count", fontsize=LABEL_FS)
    cbar.ax.tick_params(labelsize=TICK_FS)
    # ax.legend(fontsize=LEGEND_FS, loc="lower center", bbox_to_anchor=(0.5, 1.02),
    #           frameon=False, ncol=1)
    ax.legend(
    fontsize=LEGEND_FS,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.95),
    frameon=False,
    ncol=1)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)


def alpha_tie_indices(counts, alpha):
    q = np.quantile(counts, alpha)
    dist = np.abs(counts.astype(float) - q)
    return np.where(dist == dist.min())[0]


# ---------- (a) normal case: uniform sample in [0,1]^2 ----------
rng_a = np.random.default_rng()
F_a = rng_a.uniform(0.0, 1.0, size=(100, 2))
counts_a = domination_count(F_a)
G_empty = np.empty((100, 0))
sel_a = process_design(F_a, G_empty, ALPHA)
sel_idx_a = int(np.where((F_a == sel_a).all(axis=1))[0][0])
tie_a = alpha_tie_indices(counts_a, ALPHA)
print("a) n=100 uniform sample: 95th-pct count target =", np.quantile(counts_a, ALPHA),
      " tie group size =", len(tie_a), " selected idx =", sel_idx_a, "count =", counts_a[sel_idx_a])
plot_panel(F_a, counts_a, tie_a, sel_idx_a, f"{OUT}/percentile_selection_a.png")

# ---------- (b) tie-break case: 2-3 points equally close to the percentile ----------
# search seeds until a natural, small (2-3) tie group appears
for seed in range(1, 100):
    rng_b = np.random.default_rng(seed)
    F_b = rng_b.uniform(0.0, 1.0, size=(60, 2))
    counts_b = domination_count(F_b)
    tie_b = alpha_tie_indices(counts_b, ALPHA)
    if 2 <= len(tie_b) <= 3:
        break
sel_b = process_design(F_b, np.empty((60, 0)), ALPHA)
sel_idx_b = int(np.where((F_b == sel_b).all(axis=1))[0][0])
print("b) seed=", seed, " tie group size =", len(tie_b), " indices =", tie_b, " selected idx =", sel_idx_b)
plot_panel(F_b, counts_b, tie_b, sel_idx_b, f"{OUT}/percentile_selection_b.png")

# ---------- (c) degenerate case: all points on a Pareto front (all ND) ----------
rng_c = np.random.default_rng(2)
f1_c = np.sort(rng_c.uniform(0.02, 0.98, 40))
f2_c = 1.0 - f1_c
F_c = np.column_stack([f1_c, f2_c])
counts_c = domination_count(F_c)
assert (counts_c == 0).all(), "expected all-mutually-non-dominated front"
sel_c = process_design(F_c, np.empty((40, 0)), ALPHA)
sel_idx_c = int(np.where((F_c == sel_c).all(axis=1))[0][0])
tie_c = alpha_tie_indices(counts_c, ALPHA)  # == all indices, since every count is 0
print("c) all-ND: unique counts =", np.unique(counts_c), " tie group size =", len(tie_c),
      " selected idx =", sel_idx_c)
plot_panel(F_c, counts_c, None, sel_idx_c, f"{OUT}/percentile_selection_c.png", vmin=-0.1, vmax=0.1)

print("done")


a) n=100 uniform sample: 95th-pct count target = 74.19999999999999  tie group size = 1  selected idx = 26 count = 74
b) seed= 8  tie group size = 3  indices = [42 47 57]  selected idx = 42
c) all-ND: unique counts = [0]  tie group size = 40  selected idx = 18
done
